Code to detect chimeras:
1. detect potential chimeras based on the blast results ()
2. extract sequences of the potential chimeras for BLAST
3. compare the results of the BLAST for potential chimeras with initial BLAST -> detect chimeric genes
4. manually control detected chimeras
5. split the chimeric genes into two parts
6. create the gtf

In [89]:
import pandas as pd
import csv

# Load BLAST result
blast_results = pd.read_csv(
    "blast_results_V3_long.tsv", sep="\t", header=None,
    names=["qseqid", "match","qstart", "qend", "length", "evalue"]
)
# latest version of the transcriptome
gtf = pd.read_csv("transcriptome.merged.filtered.renamed.gtf", sep="\t", names=["scaffold", "source", "type", "start", "end", " ", "strand", "", "description"])
gtf = pd.read_csv("transcriptome.merged.filtered.renamed.gtf", sep="\t", names=["scaffold", "source", "type", "start", "end", " ", "strand", "", "description"])

In [90]:
def gtf_to_df(gtf_file):

    gtf_file["length"] = abs(gtf_file["end"]-gtf_file["start"] + 1)
    gtf_file[["gid","tid"]] = gtf_file["description"].str.split(";", n=1, expand=True)
    del gtf_file['source']
    del gtf_file[' ']
    del gtf_file['']
    del gtf_file['description']
    gtf_file[["","gene_id"]] = gtf_file["gid"].str.split(" ",expand=True)
    del gtf_file['gid']
    del gtf_file['']
    gtf_file[["",".","transcript_id"]] = gtf_file["tid"].str.split(" ",expand=True)
    gtf_file["transcript_id"] = gtf_file["transcript_id"].str.strip(";")
    gtf_file["transcript_id"] = gtf_file["transcript_id"].str.strip("\"")
    gtf_file["gene_id"] = gtf_file["gene_id"].str.strip("\"")
    del gtf_file['tid']
    del gtf_file['']
    del gtf_file['.']
    df = gtf_file
    return df

gtf = gtf_to_df(gtf)

In [91]:
blast_hits = pd.DataFrame()
blast_hits["transcript_id"] = blast_results["qseqid"]
blast_hits["hit_start"] = blast_results["qstart"]
blast_hits["hit_end"] = blast_results["qend"]
blast_hits["protein_name"] = blast_results["match"]

# In cases where the same fragment of a transcript hit several results - only  keep the first one
blast_hits_unique = blast_hits.drop_duplicates(subset=['transcript_id', 'hit_start', 'hit_end'])
# for each transcript get information on the best BLAST hit
df_first_protein = blast_hits_unique.groupby("transcript_id", as_index=False).first()[["transcript_id", "protein_name"]]
blast_hits_filtered = blast_hits_unique.merge(df_first_protein, on=["transcript_id", "protein_name"])

# collapse data to have info for each transcript only based on the best protein hit
collapsed_df = blast_hits_filtered.groupby('transcript_id', as_index=False).agg({
    'hit_start': 'min',
    'hit_end': 'max',
    'protein_name': 'first'
})
# reassign this data to blast_hits
blast_hits = collapsed_df
blast_hits.head(5)

,transcript_id,hit_start,hit_end,protein_name
0,SUB3.g10.t1,90,1442,sp|O15269|SPTC1_HUMAN
1,SUB3.g100.t1,1372,2364,sp|Q6ZNA5|FRRS1_HUMAN
2,SUB3.g100.t2,1078,1782,sp|Q6ZNA5|FRRS1_HUMAN
3,SUB3.g1000.t1,608,1135,sp|Q9NQR1|KMT5A_HUMAN
4,SUB3.g10004.t1,1295,2206,sp|P29323|EPHB2_HUMAN
...,...,...,...,...
10439,SUB3.g9988.t1,2,531,sp|Q02218|ODO1_HUMAN
10440,SUB3.g9989.t1,12,291,sp|Q9ULD0|OGDHL_HUMAN
10441,SUB3.g999.t1,120,1211,sp|P14060|3BHS1_HUMAN
10442,SUB3.g9993.t1,364,1131,sp|P29320|EPHA3_HUMAN


## compute nucleotide positions ##

In [92]:
gtf_sca = gtf
gtf_sca['sca'] = gtf_sca['scaffold'].str.replace("sca", "", regex=True).astype(int) # get scaffold numbers for future sorting

def compute_exon_n(group):  # Compute exon number with strand direction considered
    if group["strand"].iloc[0] == "-":
        group.loc[group["type"] == "exon", "exon_n"] = range(len(group) - 1, 0, -1)
    else:
        group.loc[group["type"] == "exon", "exon_n"] = range(1, len(group))
    group.loc[group["type"] == "transcript", "exon_n"] = 0  # Ensure transcript row is 0
    return group


gtf_sca["exon_n"] = 0
gtf_sca = gtf_sca.groupby("transcript_id", group_keys=False).apply(compute_exon_n).reset_index()

# Sort so transcript rows come first, then exons in ascending order by exon_n
gtf_sca = gtf_sca.sort_values(by=["transcript_id", "exon_n"], ascending=[True, True])
# gtf_sca

C:\Users\Dari\AppData\Local\Temp\ipykernel_12892\3242511812.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gtf_sca = gtf_sca.groupby("transcript_id", group_keys=False).apply(compute_exon_n).reset_index()


In [93]:
# Compute the total length for all rows with the same transcript_id -> relative end of the transcript
gtf_sca["relative_end"] = gtf_sca.groupby("transcript_id")["length"].transform("sum")

# Calculate relative end and start for the exons

# relative_end - cumulative sum of exon lengths
gtf_sca.loc[gtf_sca["type"] == "exon", "relative_end"] = (
    gtf_sca[gtf_sca["type"] == "exon"]
    .groupby("transcript_id")["length"]
    .cumsum()
 )

# relative_start - starts at previous exon’s relative_end + 1
gtf_sca["relative_start"] = 0
gtf_sca.loc[gtf_sca["type"] == "exon", "relative_start"] = (
    gtf_sca[gtf_sca["type"] == "exon"]
    .groupby("transcript_id")["relative_end"]
    .shift(fill_value=0) + 1
)
#  Ensure that the first exon starts at 0
first_exon_mask = gtf_sca["type"] == "exon"
gtf_sca.loc[first_exon_mask, "relative_start"] = gtf_sca.loc[first_exon_mask, "relative_start"].where(
    gtf_sca["relative_start"] > 1, 0
)

# Assign relative_end to transcript rows (total exon length)
exon_lengths = gtf_sca[gtf_sca["type"] == "exon"].groupby("transcript_id")["length"].sum()
gtf_sca.loc[gtf_sca["type"] == "transcript", "relative_end"] = gtf_sca["transcript_id"].map(exon_lengths)

# ensure that relative end and relative start values are integers and not floats
gtf_sca["relative_start"] = gtf_sca["relative_start"].astype(int)
gtf_sca["relative_end"] = gtf_sca["relative_end"].astype(int)


In [94]:
exons = gtf_sca[gtf_sca['type'] == 'exon']
filtered_exons = exons.groupby('transcript_id', as_index=False).agg({
    'relative_start': 'max',  #  max value of 'start' -> this is where the last exon starts
    'relative_end': 'min',   # min value of 'end' -> this is where the first exon ends
    'strand': 'first'
})
filtered_exons.rename(columns = {"relative_start":"last_e_start","relative_end":"first_e_end"}, inplace=True)

# filtered_exons

In [95]:
# merge the gtf with BLAST results
merged = pd.merge(blast_hits, gtf_sca[['gene_id', 'transcript_id', 'relative_start', 'relative_end', 'start', 'end', 'type', 'scaffold']], on='transcript_id', how='inner')
merged = pd.merge(merged, filtered_exons, on='transcript_id', how='inner' )
# merged

In [96]:
threshold = 1000

# Function to check if BLAST hits cover all exons
def check_coverage(row):

    exon_starts = row['last_e_start']  # start of the last exon
    exon_ends = row['first_e_end']      # end of the first exon
    blast_start = row['hit_start']
    blast_end = row['hit_end']
    true_end = row["relative_end"]
    true_start = row["relative_start"]

    # Check if all exons are fully covered
    if blast_start <= exon_ends and blast_end >= exon_starts:
        return True

    elif blast_start <= exon_ends and (true_end-blast_end) <=threshold:
        return True

    elif blast_end >= exon_starts and (blast_start-true_start) <=threshold:
        return True

    elif max([blast_start-true_start,true_end-blast_end]) <= threshold:
        return True

    else:
        return False

# Apply the coverage function to each row
merged_tr = merged.drop_duplicates(subset='transcript_id').copy()
merged_tr['fully_covered'] = merged.apply(check_coverage, axis=1)
print('Overview of fully covered transcripts:\n', merged_tr['fully_covered'].value_counts())

# merged_tr

Overview of fully covered transcripts:
 fully_covered
True     7958
False    2486
Name: count, dtype: int64


# getting remaining positions #

In [97]:
#getting remaining exon positions (all exons that are in potential chimera genes)

potential_chimeras = merged_tr[merged_tr['fully_covered'] == False]
chimeric_transcripts = potential_chimeras["transcript_id"].to_list()

print('Number of potential chimeras: ', len(set(chimeric_transcripts)))

remaining_exons = exons[exons["transcript_id"].isin(chimeric_transcripts)]

# Merge exon and blast_hits on transcript_id
merged_remaining = pd.merge(remaining_exons, blast_hits, on='transcript_id', how='left')

# Check for overlap between exon and blast hit
merged_remaining['overlap'] = (
    (merged_remaining['relative_start'] <= merged_remaining['hit_end']) & (merged_remaining['relative_end'] >= merged_remaining['hit_start'])
)
merged_remaining.head(5)

Number of potential chimeras:  2486


,index,scaffold,type,start,end,strand,length,gene_id,transcript_id,sca,exon_n,relative_end,relative_start,hit_start,hit_end,protein_name,overlap
0,1169,sca1,exon,431772,432111,-,340,SUB3.g100,SUB3.g100.t1,1,1,340,0,1372,2364,sp|Q6ZNA5|FRRS1_HUMAN,False
1,1168,sca1,exon,431532,431683,-,152,SUB3.g100,SUB3.g100.t1,1,2,492,341,1372,2364,sp|Q6ZNA5|FRRS1_HUMAN,False
2,1167,sca1,exon,431355,431446,-,92,SUB3.g100,SUB3.g100.t1,1,3,584,493,1372,2364,sp|Q6ZNA5|FRRS1_HUMAN,False
3,1166,sca1,exon,431186,431252,-,67,SUB3.g100,SUB3.g100.t1,1,4,651,585,1372,2364,sp|Q6ZNA5|FRRS1_HUMAN,False
4,1165,sca1,exon,430654,430989,-,336,SUB3.g100,SUB3.g100.t1,1,5,987,652,1372,2364,sp|Q6ZNA5|FRRS1_HUMAN,False


### split the remaining regions in two parts if needed

In [98]:
# Group by exon (transcript_id, start, end) and check if it's covered by any blast hit
coverage = merged_remaining.groupby(['scaffold', 'sca', 'transcript_id', 'start', 'end', 'exon_n', 'length'])['overlap'].any().reset_index()

# Filter exons that are not covered by any blast hit
exons_not_covered = coverage[coverage['overlap'] == False].copy()

# Detect jumps in 'exon_n' within each transcript_id group. If present indicate that only the middle part of the transcript was covered by the blast hit -> two potential chimeras
exons_not_covered["jump"] = exons_not_covered.groupby("transcript_id")["exon_n"].diff().abs() > 1

# Create a group number that increments when a jump occurs
exons_not_covered["group"] = exons_not_covered.groupby("transcript_id")["jump"].cumsum()
# adjust the transcript ids so that there are two possible areas where gene might end up being a chimera
exons_not_covered['transcript_id_n'] = exons_not_covered['group'].apply(lambda x: '.sta' if x == 0 else '.end')
exons_not_covered['transcript_id_n'] = exons_not_covered['transcript_id'] + exons_not_covered['transcript_id_n']

exons_not_covered = exons_not_covered.drop(columns=["jump", "group", 'exon_n', 'length'])
exons_not_covered.head(5)

,scaffold,sca,transcript_id,start,end,overlap,transcript_id_n
0,sca1,1,SUB3.g100.t1,427610,428027,False,SUB3.g100.t1.sta
9,sca1,1,SUB3.g100.t1,430369,430520,False,SUB3.g100.t1.end
10,sca1,1,SUB3.g100.t1,430654,430989,False,SUB3.g100.t1.end
11,sca1,1,SUB3.g100.t1,431186,431252,False,SUB3.g100.t1.end
12,sca1,1,SUB3.g100.t1,431355,431446,False,SUB3.g100.t1.end


In [99]:
# #to just get the positions where we should start cutting
nucl_positions = exons_not_covered.groupby('transcript_id_n', as_index=False).agg({
    'scaffold': 'first',
    'transcript_id': 'first',
    'start': 'min',  # Take the max value of 'start' -> this is where the last exon starts
    'end': 'max'   # Take the min value of 'end' -> this is where the first exon ends

}).merge(remaining_exons[["transcript_id","strand"]].drop_duplicates(subset="transcript_id"),how="inner",on="transcript_id")

nucl_positions['transcript_id'] = nucl_positions['transcript_id_n']

nucl_positions['type'] = 'transcript'
nucl_positions = nucl_positions.drop(columns=['transcript_id_n'])

nucl_positions["strand"].value_counts()

strand
+    1787
-    1749
Name: count, dtype: int64

### export to gtf data

In [100]:
# adjust exons df so that it can be merged onto final gtf
exons_not_covered['transcript_id'] = exons_not_covered['transcript_id_n']
exons_not_covered['type'] = 'exon'

exons_not_covered = exons_not_covered.merge(nucl_positions[["transcript_id","strand"]].drop_duplicates(subset="transcript_id"),how="inner",on="transcript_id")

exons_not_covered.head(3)

,scaffold,sca,transcript_id,start,end,overlap,transcript_id_n,type,strand
0,sca1,1,SUB3.g100.t1.sta,427610,428027,False,SUB3.g100.t1.sta,exon,-
1,sca1,1,SUB3.g100.t1.end,430369,430520,False,SUB3.g100.t1.end,exon,-
2,sca1,1,SUB3.g100.t1.end,430654,430989,False,SUB3.g100.t1.end,exon,-
3,sca1,1,SUB3.g100.t1.end,431186,431252,False,SUB3.g100.t1.end,exon,-
4,sca1,1,SUB3.g100.t1.end,431355,431446,False,SUB3.g100.t1.end,exon,-
...,...,...,...,...,...,...,...,...,...
18495,sca9,9,SUB3.g14428.t1.sta,4357171,4357299,False,SUB3.g14428.t1.sta,exon,+
18496,sca9,9,SUB3.g14428.t1.sta,4357357,4357445,False,SUB3.g14428.t1.sta,exon,+
18497,sca9,9,SUB3.g14428.t1.sta,4357505,4357621,False,SUB3.g14428.t1.sta,exon,+
18498,sca9,9,SUB3.g14428.t1.sta,4357680,4357807,False,SUB3.g14428.t1.sta,exon,+


In [101]:
nucl_pos_and_exons = pd.concat([nucl_positions, exons_not_covered], axis=0, ignore_index=True)
nucl_pos_and_exons = nucl_pos_and_exons.drop(columns=["sca", "overlap", "transcript_id_n"])
nucl_pos_and_exons['transcript_id_n'] = nucl_pos_and_exons['transcript_id']
nucl_pos_and_exons['transcript_id'] = nucl_pos_and_exons['transcript_id'].str[:-4]

,scaffold,transcript_id,start,end,strand,type
0,sca1,SUB3.g100.t1.end,430369,432111,-,transcript
1,sca1,SUB3.g100.t1.sta,427610,428027,-,transcript
2,sca1,SUB3.g100.t2.end,430654,432111,-,transcript
3,sca1,SUB3.g100.t2.sta,427610,428027,-,transcript
4,sca30,SUB3.g10004.t1.sta,43509,45295,+,transcript
...,...,...,...,...,...,...
22031,sca9,SUB3.g14428.t1.sta,4357171,4357299,+,exon
22032,sca9,SUB3.g14428.t1.sta,4357357,4357445,+,exon
22033,sca9,SUB3.g14428.t1.sta,4357505,4357621,+,exon
22034,sca9,SUB3.g14428.t1.sta,4357680,4357807,+,exon


In [103]:
# finalize the gtf for potential chimeras
raw_gtf = pd.read_csv("transcriptome.merged.filtered.renamed.gtf",sep="\t",names=["scaffold", "source", "type","start","end", " ","strand","","description"])

df = nucl_pos_and_exons.merge(gtf[["gene_id","transcript_id"]].drop_duplicates(subset="transcript_id"),how="inner",on="transcript_id")

df["description"] = 'gene_id "' + df["gene_id"] + '"; transcript_id "' +df["transcript_id_n"] + '";'

del df["gene_id"]
df[""] = '.'
df[" "] = '.'
df["source"] = 'StringTie'
df['sca'] = df['scaffold'].str.replace("sca", "", regex=True).astype(int)
df['transcript'] = df['transcript_id'].str.split('.t').str[-1].astype(int)
df['type_order'] = df['type'].map({'transcript': 0, 'exon': 1})

# create a new df to sort transcripts by the scaffolds and start positions between themselves so that it can later be applied to the exons too
sorted_tr = df[df['type'] == 'transcript']
sorted_tr = sorted_tr.sort_values(by=['sca', 'start', 'transcript', 'type_order'], ascending=[True, True, True, True]).reset_index(drop=True)
sorted_tr['sort_order'] = sorted_tr.index + 1
sorted_tr = sorted_tr[['transcript_id_n', 'sort_order']]

# sort the resulting dataframe
sorted_final_df = pd.merge(df, sorted_tr, on='transcript_id_n', how='inner')
sorted_final_df = sorted_final_df.sort_values(by=['sort_order', 'start', 'type_order'], ascending=[True, True, True]).reset_index(drop=True)
sorted_final_df['transcript_id'] = sorted_final_df['transcript_id_n']
sorted_final_df = sorted_final_df[raw_gtf.columns.to_list()]
sorted_final_df.head(5)

,scaffold,source,type,start,end,,strand,,description
0,sca1,StringTie,transcript,83137,84213,.,-,.,"gene_id ""SUB3.g16""; transcript_id ""SUB3.g16.t1..."
1,sca1,StringTie,exon,83137,83177,.,-,.,"gene_id ""SUB3.g16""; transcript_id ""SUB3.g16.t1..."
2,sca1,StringTie,exon,83241,83307,.,-,.,"gene_id ""SUB3.g16""; transcript_id ""SUB3.g16.t1..."
3,sca1,StringTie,exon,83371,83466,.,-,.,"gene_id ""SUB3.g16""; transcript_id ""SUB3.g16.t1..."
4,sca1,StringTie,exon,83523,83621,.,-,.,"gene_id ""SUB3.g16""; transcript_id ""SUB3.g16.t1..."
...,...,...,...,...,...,...,...,...,...
22031,sca784,StringTie,transcript,12898,14777,.,+,.,"gene_id ""SUB3.g12934""; transcript_id ""SUB3.g12..."
22032,sca784,StringTie,exon,12898,13027,.,+,.,"gene_id ""SUB3.g12934""; transcript_id ""SUB3.g12..."
22033,sca784,StringTie,exon,13117,13314,.,+,.,"gene_id ""SUB3.g12934""; transcript_id ""SUB3.g12..."
22034,sca784,StringTie,exon,13419,14351,.,+,.,"gene_id ""SUB3.g12934""; transcript_id ""SUB3.g12..."


In [104]:
print(sorted_final_df.drop_duplicates(subset='description').shape)
sorted_final_df.to_csv('potential_chimeras_f_param_strand_ds_250220.gtf', sep='\t', header=False, index=False, quoting=csv.QUOTE_NONE, escapechar='\\')

(3536, 9)


### Splitting chimeric genes
following steps are run through after the reciprocal blast for potential chimeras is performed

In [106]:
blast_full = pd.read_excel("updated_Sd_Hs.xlsx")

chimera_blast = pd.read_excel("sponge_chimer_human_rb_e30_f_coord_strand_ds_250220.xlsx")

chimera_blast['Sponge gene full'] = chimera_blast['Sponge gene']
chimera_blast['Sponge gene'] = chimera_blast['Sponge gene'].str[:-4] # get the initial transcript ids which the potential chimera was split from
blast_full['blast_res_old'] = blast_full['blast_res']
del blast_full['blast_res']
del blast_full["e-value_y"]

# merge initial and new BLAST results
df = pd.merge(blast_full, chimera_blast, on='Sponge gene', how='right')[["Sponge gene full","Protein ID_x","Gene Symbol_x","blast_res","Gene Symbol_y"]]

In [107]:
df_success = df[df['blast_res'] == 'success']
# filter so that only rows where old and new blast results are not identical
df = df_success[df_success['Gene Symbol_x'] != df_success['Gene Symbol_y']]
print(df.shape)

def count_exact_matches(s1, s2):
    s1, s2 = str(s1) if s1 is not None else '', str(s2) if s2 is not None else ''
    res = sum(c1 == c2 for c1, c2 in zip(s1, s2))
    return res/max(len(s1),len(s2))

# Apply function to DataFrame
df = df.copy()
df['exact_match_count'] = df.apply(lambda x: count_exact_matches(x['Gene Symbol_x'], x['Gene Symbol_y']), axis=1)
# df

(232, 5)


In [108]:
# save the dataframe to manually check detected chimeras (in case the gene symbols are sufficiently different but the actual genes are related)
# df[['Sponge gene full', 'Gene Symbol_x', 'Gene Symbol_y', 'exact_match_count']].to_excel('potential_chimera_blast_comparison_f_ds_250224.xlsx', index=False)

In [109]:
# only leave sponge genes where exact_match_count < 0.5
filtered_sponge_genes = df.loc[df['exact_match_count'] < 0.5, 'Sponge gene full']

# Remove genes that were manually determined to not be chimera
filtered_sponge_genes = filtered_sponge_genes[~filtered_sponge_genes.isin(["SUB3.g1241.t1.end", "SUB3.g393.t1.sta"])]

# only leave the rows from the potential chimeras gtf that belong to confirmed chimeras
filtered_final = sorted_final_df[sorted_final_df['description'].str.contains('|'.join(filtered_sponge_genes), na=False, regex=True)]
# rename the transcripts to differentiate them
filtered_final.loc[:, 'description']  = filtered_final['description'].str.replace('SUB3', 'SUB4', regex=True)

print('number of transcripts : ', len(filtered_final[filtered_final['type'] == 'transcript']))
print('number of genes: ', len(set(filtered_final['description'].str.split('; ').str[0].to_list())))
# filtered_final  # filtered out chimeras (already split from the rest)

number of transcripts :  228
number of genes:  166


In [110]:
# extract non-chimeric genes
filtered_sponge_genes_short = [gene[:-4] for gene in filtered_sponge_genes]

non_chimeras = gtf_sca[~gtf_sca['transcript_id'].isin(filtered_sponge_genes_short)]
del non_chimeras['sca']
del non_chimeras['exon_n']
# non_chimeras

In [112]:
# extract the second part of chimera
# get all rows from gtf from the transcripts that were determined to be chimeras
chimeric_g = gtf_sca[gtf_sca['transcript_id'].isin(filtered_sponge_genes_short)].copy()
chimeric_g[['gene_number', 'transcript_suffix']] = chimeric_g['transcript_id'].str.extract(r'\.g(\d+)\.t(\d+)')

# remove exons that are selected as separate genes
filtered_final_m = filtered_final[['scaffold', 'type', 'start', 'end', 'description']].copy()
filtered_final_m[['gene_number', 'transcript_suffix']] = filtered_final_m['description'].str.extract(r'\.g(\d+)\.t(\d+)')

# anti-join - leave the rows that have only the values that are not identical between dataframes
gtf_chimeric_g = chimeric_g.merge(filtered_final_m, left_on=['scaffold', 'type', 'start', 'end', 'gene_number', 'transcript_suffix'], right_on=['scaffold', 'type', 'start', 'end', 'gene_number', 'transcript_suffix'], how='left', indicator=True)

gtf_chimeric_g = gtf_chimeric_g[gtf_chimeric_g['_merge'] == 'left_only'].drop(columns=['sca', 'exon_n', 'gene_number', 'transcript_suffix', 'description', '_merge'])

# adjust transcript coordinates
transcript_bounds = gtf_chimeric_g[gtf_chimeric_g['type'] == 'exon'].groupby('transcript_id').agg({'start': 'min', 'end': 'max'}).reset_index()

gtf_chimeric_g = gtf_chimeric_g.merge(
    transcript_bounds,
    on='transcript_id',
    how='left',
    suffixes=('', '_corrected')
)

# Replace only where type == 'transcript'
gtf_chimeric_g.loc[gtf_chimeric_g['type'] == 'transcript', ['start', 'end']] = \
    gtf_chimeric_g.loc[gtf_chimeric_g['type'] == 'transcript', ['start_corrected', 'end_corrected']].values
gtf_chimeric_g.drop(columns=['start_corrected', 'end_corrected'], inplace=True)

gtf_chimeric_g.head(5)

,index,scaffold,type,start,end,strand,length,gene_id,transcript_id,relative_end,relative_start
0,119825,sca30,transcript,280021,286007,-,26561,SUB3.g10033,SUB3.g10033.t1,8895,0
1,119869,sca30,exon,285801,286007,-,207,SUB3.g10033,SUB3.g10033.t1,207,0
2,119868,sca30,exon,285651,285678,-,28,SUB3.g10033,SUB3.g10033.t1,235,208
3,119867,sca30,exon,285326,285378,-,53,SUB3.g10033,SUB3.g10033.t1,288,236
4,119866,sca30,exon,285176,285227,-,52,SUB3.g10033,SUB3.g10033.t1,340,289


In [113]:
# merge all parts into one df
merged_chim_nonchim = pd.concat([non_chimeras, gtf_chimeric_g], axis=0)
merged_chim_nonchim["description"] = 'gene_id "' + merged_chim_nonchim["gene_id"] + '"; transcript_id "' +merged_chim_nonchim["transcript_id"] + '";'

del merged_chim_nonchim["gene_id"]
del merged_chim_nonchim["transcript_id"]
merged_chim_nonchim[""] = '.'
merged_chim_nonchim[" "] = '.'
merged_chim_nonchim["source"] = 'StringTie'
merged_chim_nonchim = merged_chim_nonchim[raw_gtf.columns.to_list()]

# merge with confirmed chimeras
merged_final = pd.concat([merged_chim_nonchim, filtered_final], axis=0)


merged_final.shape

(166769, 9)

In [ ]:
# write to intermediate gtf
merged_final.to_csv('sponge_chimer_human_rb_e30_curated_250220.gtf', sep='\t', header=False, index=False, quoting=csv.QUOTE_NONE, escapechar='\\')